[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/exercise11-guided-r-colab-2026-11-15/notebooks/Assignment_11_R_Gene_Expression_and_Volcano_Plots_in_Colab.ipynb)

[Open Exercise 11 in Google Colab](https://colab.research.google.com/github/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/exercise11-guided-r-colab-2026-11-15/notebooks/Assignment_11_R_Gene_Expression_and_Volcano_Plots_in_Colab.ipynb) · [View R notebook on GitHub](https://github.com/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/exercise11-guided-r-colab-2026-11-15/notebooks/Assignment_11_R_Gene_Expression_and_Volcano_Plots_in_Colab.ipynb)

# Assignment 11 — R Gene Expression Analysis and Volcano Plots in Colab
**30 points · approximately 120–150 minutes · native R notebook**

Use the **Open in Colab** button on Canvas or above. Choose **File → Save a copy in Drive** before editing; the shared GitHub notebook remains the course master. Confirm **Runtime → Change runtime type → R**, use CPU, and save a copy to your own Drive. Run all cells, then edit the Markdown response tables A–F. All computation uses base R: no extra packages, Drive mount, or external data are required. If the notebook opens in Python, switch to R before running.

The toy data and metadata are embedded below and are also supplied as CSV files. **240 simulated genes, 8 independent simulated biological samples: 4 controls and 4 treated.** Values are simulated normalized **log2 expression**, similar to a continuous expression teaching dataset. They are **not raw RNA-seq counts**, TPM, real measurements, or results from an actual organism. They have no gene annotations.

We use gene-wise Welch t-tests on these continuous log2 values, followed by Benjamini–Hochberg (BH) adjustment. This is a transparent teaching model for effect size, uncertainty, multiple testing, and plotting. **Do not use this code as a raw-count RNA-seq pipeline.** Raw counts require an appropriate count model, normalization, and design such as a DESeq2 workflow. Unlike the lecture's unreplicated 12×6 time-course matrix, this new toy dataset has replication for the statistical lesson.

**Required hands-on edits:** change the numeric values or logical setting on lines marked `# EDIT THIS LINE`. The formatted cell immediately above tells you exactly what to change, what to keep fixed, which cell to rerun, and what to record. Predict before running and explain the before/after result. All activities fit within the existing 30-point rubric.

## Why use R for gene-expression analysis?
R gives us direct access to a specialist statistical and genomics ecosystem. For a gene-expression project, this can make it a more convenient starting point than rebuilding the workflow from general-purpose Python tools. The advantage is the available methods and their integration, not a claim that R always computes better results.

| R strength | What it adds to a gene-expression project | What we use today |
|---|---|---|
| **Bioconductor: DESeq2 and edgeR** | Count-data models for differential expression, including estimation of variability across genes. DESeq2 explicitly models the mean–variance relationship with a negative-binomial model. | We discuss the role of these tools; we do **not** install or run them in this small continuous-data exercise. |
| **limma** | Linear-model methods for microarray and other omics expression data; a route to more complex contrasts and study designs. | We begin with a transparent two-group Welch test so you can see where effects and p-values come from. |
| **Statistical objects and functions** | Model fitting, contrasts, hypothesis tests, multiple-testing adjustment, and diagnostic summaries can be combined in a statistical workflow. | `t.test()`, `p.adjust(method="BH")`, and `prcomp()`. |
| **ggplot2** | Layered graphics: map data to axes/colors, then add points, labels, thresholds, and panels consistently. Useful for extending volcano and MA plots. | Base-R graphics keep setup small; the same plotting ideas transfer to ggplot2. |

Python remains valuable for sequence processing, automation, scientific computing, and machine learning. Both languages can draw volcano plots and perform statistical analyses. Here R is useful because learning its conventions prepares you to use the specialist R packages directly. Choose the tool for the dataset, model, and workflow, and check assumptions in either language.

**The important boundary:** this notebook's toy values are continuous normalized log2 expression. Gene-wise Welch tests illustrate statistical ideas; they do not supply the count normalization, dispersion estimation, or design modeling of a full RNA-seq count workflow. Never feed these log2 values into DESeq2 as raw counts. You do not need to install Bioconductor or ggplot2 for today's required work.

Sources: [DESeq2](https://bioconductor.org/packages/release/bioc/html/DESeq2.html), [edgeR](https://bioconductor.org/packages/release/bioc/html/edgeR.html), [limma](https://bioconductor.org/packages/release/bioc/html/limma.html), [ggplot2](https://ggplot2.tidyverse.org/).

**R reading guide:** `<-` assigns a value; `#` begins a comment; `TRUE` and `FALSE` are logical values (capital letters matter). For example, in `noise_sd <- 0.35  # EDIT THIS LINE`, change `0.35` and leave the name, arrow, and comment in place.

## Goals and key terms
Read an R expression matrix and sample table; inspect sample PCA; calculate treatment-minus-control effects; distinguish raw and adjusted p-values; construct and interpret volcano and MA plots; compare effect thresholds and replicate variability.

Key terms: biological replicate, observation, feature, log2 expression, log2 fold change, Welch test, p-value, BH-adjusted p-value, false discovery rate, PCA, volcano plot, MA plot, effect size.

Lecture connections: *Experimental design before expression analysis; Counts, abundance and transformed values; Effect size and statistical evidence; PCA: observations and features; PCA loadings and interpretation; Exploration, inference and prediction; Expression patterns and biological claims*. This exercise extends those concepts to R and volcano plots; it does not assign metabolic reconstruction.

In [ ]:
# Check that this is an R runtime.
print(R.version.string)
result_dir <- "assignment_11_results"
dir.create(result_dir, showWarnings = FALSE)
expr <- as.matrix(read.csv(text = "gene,C1,C2,C3,C4,T1,T2,T3,T4
Gene_001,6.76334,7.34536,7.1662,6.68506,9.41788,8.84678,9.1267,8.7929
Gene_002,9.49208,9.58167,9.60524,9.38159,11.19104,11.17091,10.54827,11.53989
Gene_003,7.72925,7.10214,7.88861,7.13172,8.6996,9.68383,9.34627,10.17707
Gene_004,6.66979,5.61114,6.50852,5.61018,7.38484,8.60216,7.7403,8.10115
Gene_005,8.9803,10.94801,9.24519,9.37876,11.99353,11.27585,11.6747,12.21919
Gene_006,10.77549,11.59263,11.3132,10.68378,13.21617,12.46159,12.36428,12.86933
Gene_007,10.41843,10.51124,10.97032,10.81252,13.64886,12.84788,12.57881,13.25815
Gene_008,9.84893,9.52365,9.3533,8.6895,11.2692,11.17369,11.12839,11.01699
Gene_009,6.91376,6.1346,6.84578,6.4345,9.00221,7.7772,9.2108,8.22915
Gene_010,9.40078,9.42082,8.93039,9.78596,11.48131,11.45809,11.22621,11.14015
Gene_011,11.2778,11.34535,11.53464,11.77773,12.9518,12.44924,12.41764,13.07856
Gene_012,8.52194,9.29098,9.09411,8.79369,10.17312,10.07286,9.65128,9.78862
Gene_013,9.6137,10.45624,9.5626,10.17538,11.28643,11.62927,11.60422,11.73665
Gene_014,6.34759,7.01742,6.98099,6.74057,8.95268,8.73043,9.15599,8.63062
Gene_015,9.42546,8.98693,9.67899,9.71679,11.65125,12.21289,11.72457,11.56979
Gene_016,8.43877,7.94334,8.16721,8.68073,10.2702,10.10175,10.71113,10.83657
Gene_017,9.12402,9.79359,8.76814,9.57449,11.78829,12.0419,11.46126,10.98878
Gene_018,11.35503,11.01806,10.40754,10.95208,12.45459,13.10102,12.97274,12.33735
Gene_019,6.27254,6.6322,7.16173,6.60628,8.16803,8.32893,7.63655,8.96818
Gene_020,8.22653,9.01707,8.56817,8.83016,10.53117,10.7052,10.60116,10.89113
Gene_021,8.50656,8.53823,8.69035,8.17844,10.19153,10.85273,10.68826,10.08691
Gene_022,9.57606,10.05989,10.48517,9.93441,11.84415,11.25305,11.49115,11.95369
Gene_023,10.96997,11.48869,11.13658,10.56885,12.04132,11.57341,12.52999,12.56728
Gene_024,5.94454,6.41415,7.70777,6.45056,8.34271,8.49474,8.22701,9.25203
Gene_025,8.15926,9.39874,7.937,8.12925,9.05688,9.84393,10.72775,11.00157
Gene_026,7.36945,7.72244,7.24864,6.97946,9.62663,10.55007,9.39111,9.7289
Gene_027,6.60285,7.92649,7.43954,8.64489,9.66096,8.86805,9.49819,9.22876
Gene_028,9.27567,9.39656,10.07722,8.89371,12.22816,12.29103,12.183,11.22213
Gene_029,9.53429,10.20988,10.28382,10.55654,11.51525,12.15405,11.56198,12.07442
Gene_030,10.83659,10.62209,9.98625,9.82206,11.67979,12.22118,11.68229,11.39595
Gene_031,9.51795,9.35865,9.99195,10.03777,8.61906,8.03137,7.9412,6.68798
Gene_032,6.36752,6.30031,6.37279,5.92602,3.24748,3.96379,3.77158,4.34171
Gene_033,10.08051,10.03996,9.93375,10.05875,8.23098,7.65813,7.00611,7.33407
Gene_034,11.36983,9.71951,10.67271,10.60198,8.07417,9.23179,8.19336,8.31711
Gene_035,9.19479,9.21397,9.48988,9.89777,7.12352,7.35366,8.6956,8.18116
Gene_036,6.52567,7.1888,6.86298,7.43913,4.16077,5.03926,5.00715,5.33626
Gene_037,9.17696,8.9946,8.18689,8.70256,6.56981,6.91365,6.86002,7.57112
Gene_038,7.58557,8.79595,7.87713,9.37178,5.84073,5.69109,5.84338,6.28705
Gene_039,9.71327,7.98965,8.92594,8.57549,6.67178,6.94088,5.76637,6.33615
Gene_040,11.12909,10.25791,11.23851,10.20826,9.06977,8.67523,8.07321,8.01639
Gene_041,6.44295,6.49664,6.70614,6.08604,4.44961,4.51773,3.8393,5.04266
Gene_042,8.60824,8.8552,8.43964,9.35118,7.08067,7.66471,7.32452,6.85866
Gene_043,10.55747,9.93001,9.47462,10.46338,7.89674,8.09974,8.01003,8.44951
Gene_044,6.29159,5.82695,5.93516,6.34682,4.68314,3.34634,3.35755,4.47073
Gene_045,10.98823,10.20558,10.70239,11.4201,8.40515,9.18683,8.30724,8.61721
Gene_046,9.98885,8.89056,9.61162,8.68201,7.21866,7.80701,7.84738,7.93443
Gene_047,9.91348,9.08431,9.64832,9.29242,7.05894,7.62139,7.40141,6.36383
Gene_048,10.75438,10.61988,11.23442,10.07155,8.56224,8.726,9.05023,8.41073
Gene_049,5.6522,6.04849,5.55442,6.09996,4.86825,4.39123,3.46133,3.6799
Gene_050,10.7008,9.36041,10.46332,10.02173,7.51172,7.57396,7.72165,7.38723
Gene_051,7.45492,8.10001,7.50118,7.16242,5.76096,5.37533,5.07334,5.05148
Gene_052,9.57338,9.47363,9.33015,9.1638,7.44176,8.16125,6.88197,6.69302
Gene_053,6.19684,5.50629,6.17111,5.84996,4.10966,5.02123,3.55285,3.47948
Gene_054,5.78431,5.93704,5.84706,6.00097,4.63321,4.16441,4.6658,4.36675
Gene_055,10.18655,8.50114,9.06281,9.20021,7.0997,7.8201,7.59057,8.28436
Gene_056,10.47856,10.14577,9.85273,9.6127,8.78947,7.47863,8.04653,7.69301
Gene_057,8.53707,7.87403,7.86858,7.57679,5.76125,6.80319,6.5267,6.46107
Gene_058,9.00811,7.71078,8.92315,8.24054,7.09208,6.58019,6.579,6.75719
Gene_059,8.15483,7.82823,7.56522,7.88453,6.02041,6.80707,5.72863,5.9755
Gene_060,7.82969,8.48273,7.1453,7.95263,5.167,5.14177,6.10829,5.96362
Gene_061,8.53673,8.54707,8.54564,8.63258,9.01451,8.66925,9.0013,8.94507
Gene_062,9.18562,9.02967,9.07758,9.44089,9.48335,9.62955,9.45158,9.40361
Gene_063,8.30836,8.49311,8.42786,8.27807,8.754,8.69132,8.79948,9.0183
Gene_064,9.59434,9.62663,9.36884,9.38163,10.07937,9.98391,9.76679,9.86562
Gene_065,8.22774,8.38682,8.25803,8.26799,8.63801,8.78676,8.80438,8.72701
Gene_066,8.1661,8.40459,8.36944,8.38741,8.72994,8.74943,8.6367,9.03328
Gene_067,7.17734,7.29126,7.28318,7.14859,7.52371,7.32477,7.68293,7.62449
Gene_068,8.42316,8.33431,8.30594,8.44512,8.71196,8.75135,8.84874,8.63418
Gene_069,10.91167,10.89259,10.96878,10.98232,11.25625,11.2954,11.33428,11.29318
Gene_070,8.15146,8.2518,8.31514,8.27497,8.56666,8.62844,8.54326,8.68605
Gene_071,11.1251,10.95417,10.97165,11.1766,11.33612,11.24397,11.54343,11.24654
Gene_072,7.10664,7.24056,7.08072,7.06237,7.29358,7.58912,7.38518,7.53963
Gene_073,6.43855,6.38259,6.61322,6.59496,6.87468,7.07769,6.80405,6.80589
Gene_074,9.9784,10.07892,10.04904,9.97656,10.28601,10.26742,10.25157,10.17678
Gene_075,6.32214,6.4019,6.42863,6.53757,6.72065,6.80524,6.7425,6.87219
Gene_076,8.37449,8.51109,8.38829,8.46009,8.74675,8.77933,8.83981,8.91744
Gene_077,7.14179,7.04248,6.99464,7.21014,7.63355,7.31471,7.3597,7.49487
Gene_078,6.69394,7.01762,6.83875,6.98381,7.23436,7.17749,7.29,7.44116
Gene_079,8.40208,8.36248,8.12832,8.50909,8.90683,8.76279,8.65739,8.90565
Gene_080,10.76086,10.64069,10.80082,11.06436,11.42654,11.26849,11.09795,11.14388
Gene_081,7.60937,5.818,4.82302,7.69925,7.58981,8.96473,8.64168,8.38653
Gene_082,7.62686,12.049,8.74675,9.12696,9.31743,11.15759,9.62591,10.02805
Gene_083,10.93888,12.72951,11.35645,9.27148,13.85406,13.50042,9.13294,12.20577
Gene_084,6.77622,10.25411,10.6878,11.21794,11.0401,12.70771,13.04321,7.45086
Gene_085,7.0516,9.09507,5.49588,8.54842,15.09656,13.26461,14.64993,9.61497
Gene_086,8.43813,8.61195,10.25625,11.1435,9.03143,11.50328,10.58716,11.85755
Gene_087,5.86439,6.00089,9.23365,6.19871,10.12337,10.50447,7.23341,9.02314
Gene_088,11.9646,6.48184,9.30966,8.58282,11.84577,10.68128,9.85417,13.00219
Gene_089,7.43039,8.61002,8.29307,10.99957,10.1989,4.96256,11.84171,11.08334
Gene_090,8.15599,7.24329,7.90509,6.98449,7.62703,9.6397,9.71343,9.94638
Gene_091,10.30588,10.98548,11.60026,10.03785,10.83367,11.26134,10.46599,11.56195
Gene_092,9.21629,9.78133,9.98177,10.78883,9.19235,9.9125,9.7188,9.57967
Gene_093,8.3368,8.15996,8.31898,8.2837,9.0322,9.40618,8.53538,8.34127
Gene_094,7.88896,8.45793,8.39561,7.89659,8.90169,8.23123,8.26413,7.86774
Gene_095,9.25151,10.83799,10.57828,10.34391,11.01704,10.99139,10.59873,11.35717
Gene_096,8.29908,9.14987,9.03356,8.04645,8.18629,8.55268,8.47738,8.45951
Gene_097,7.6419,9.07655,9.03513,8.15435,9.22233,8.99613,8.12756,8.33517
Gene_098,10.4462,9.85981,10.46129,9.97971,10.63743,10.46148,10.43206,10.51978
Gene_099,8.92321,8.29437,9.34626,8.03464,8.03269,9.66821,7.83396,8.58659
Gene_100,10.08062,10.85789,10.40645,9.92785,10.30715,9.44525,9.5677,10.5487
Gene_101,10.84568,11.53332,11.37793,10.84362,11.01382,10.79111,10.61793,9.93146
Gene_102,7.8701,8.97543,8.40031,8.06398,8.28096,9.01092,8.86066,7.91904
Gene_103,7.16829,6.70846,7.16931,7.1271,7.36868,7.49307,8.08738,7.4878
Gene_104,9.56858,9.18725,9.41344,9.70667,9.38038,10.10885,8.52994,9.47638
Gene_105,9.72541,10.1919,9.43517,9.92922,10.75485,10.01416,9.7547,9.85017
Gene_106,11.03824,10.72096,10.109,9.81229,9.89856,9.93136,9.97228,10.15001
Gene_107,10.9604,11.80953,10.35755,10.47237,10.15901,11.15452,10.75028,11.30176
Gene_108,6.35592,6.44735,6.59217,6.6281,7.24143,7.13305,6.25758,6.5985
Gene_109,9.67752,11.07655,10.74687,11.05204,10.73698,11.0908,10.42885,10.30457
Gene_110,10.6674,11.15454,10.63628,10.92302,10.70577,11.6604,11.58839,10.97283
Gene_111,11.31947,10.74,10.64122,10.6131,11.48801,11.06038,10.68607,10.42123
Gene_112,8.79694,7.45235,8.47544,8.39498,8.30086,8.31059,8.12459,8.18838
Gene_113,10.42133,9.95953,10.25539,8.832,9.54301,8.99907,9.67933,9.91549
Gene_114,6.52016,6.7237,6.48213,5.98217,6.08936,6.81037,6.61598,6.31341
Gene_115,9.91504,9.60599,9.11002,9.52685,9.24053,10.27469,9.09481,9.77896
Gene_116,6.88275,6.55977,6.99353,7.15556,7.40686,7.60468,7.21507,6.95441
Gene_117,7.78545,7.35603,7.95185,7.80211,8.33694,7.32916,7.80324,7.31924
Gene_118,9.17652,8.71138,8.80553,9.23772,10.06716,9.75223,9.04143,9.62665
Gene_119,9.22224,8.97193,8.73656,9.10739,9.29245,9.81241,9.13005,9.00123
Gene_120,9.98912,9.81549,9.18685,9.49911,8.46316,9.0352,9.1592,10.67711
Gene_121,9.62082,8.21786,8.242,8.36381,8.92505,8.07017,9.74292,8.16075
Gene_122,9.07267,8.06629,8.09506,8.82352,8.64097,8.86697,8.26881,8.24265
Gene_123,11.20746,9.86901,10.22069,11.10951,10.22561,11.24879,11.06102,10.21811
Gene_124,9.32686,9.78996,9.17761,9.2006,9.81014,9.79826,9.63895,9.33804
Gene_125,9.10629,8.33247,8.97208,9.30507,8.49326,9.63183,9.18762,8.00647
Gene_126,6.73958,7.46333,6.26704,6.89165,7.26636,7.22275,7.78633,7.04461
Gene_127,6.06373,6.21509,6.6688,5.55082,6.7087,5.67504,6.03426,5.7931
Gene_128,7.38531,7.69045,7.79852,7.60124,7.55809,7.80059,7.41289,7.27557
Gene_129,6.67704,6.47007,6.10807,6.3768,6.80149,6.83957,7.07993,6.17337
Gene_130,10.92363,10.50395,10.11118,10.64153,10.74467,10.76443,10.74024,10.75336
Gene_131,11.81023,11.39277,9.19774,10.85006,10.43207,10.11651,10.6961,10.20978
Gene_132,11.00018,11.04053,10.22161,10.03765,11.81071,11.05563,11.26035,10.81758
Gene_133,9.89926,9.81388,9.56234,9.92487,10.38094,10.41707,10.02062,9.9299
Gene_134,9.86009,10.49969,9.26894,9.67564,9.70229,9.93349,9.62063,9.81659
Gene_135,7.90877,7.33011,7.84841,8.66768,8.67693,7.28281,8.74688,8.02821
Gene_136,10.59247,9.17905,9.9191,9.45426,9.58938,10.10515,10.51285,9.53778
Gene_137,7.48734,6.08512,7.08038,6.41168,7.69431,6.78538,7.13893,6.61963
Gene_138,8.42238,9.21456,8.98124,9.63917,8.80372,8.31271,9.11332,8.42793
Gene_139,8.04585,7.6936,7.91892,8.03213,8.37757,7.69959,8.20521,6.95468
Gene_140,8.12031,8.49743,6.86725,7.5438,7.41864,7.80904,7.87296,7.47551
Gene_141,8.48047,9.38174,8.21294,8.23198,8.56966,8.96204,8.5317,8.56755
Gene_142,10.2769,10.80809,10.78994,10.00084,11.51069,10.55611,10.2632,11.01479
Gene_143,10.97353,10.84137,10.42882,10.87757,10.31875,10.64699,10.49954,10.28682
Gene_144,9.19737,9.47959,9.51707,9.24073,10.0204,10.07102,9.30912,10.1104
Gene_145,7.02654,7.45427,7.7873,7.47856,7.48813,7.58723,7.86258,7.07495
Gene_146,7.81478,7.8388,8.24866,8.84993,7.97203,7.74879,7.93944,8.44298
Gene_147,6.30696,5.628,6.30314,5.48985,5.62554,6.42233,5.90626,5.77642
Gene_148,9.78232,11.40095,10.39886,10.88851,11.44171,11.33167,10.81176,9.59836
Gene_149,9.9724,9.53977,9.23991,8.20114,9.14386,8.91488,8.67791,9.6642
Gene_150,5.509,6.82592,6.01048,6.21118,5.87581,5.93002,5.70253,6.59315
Gene_151,6.31523,5.82342,7.21426,6.88698,7.51443,7.90761,6.95771,6.41773
Gene_152,10.48212,10.3803,10.56144,10.77993,10.9036,10.43202,10.79103,10.37784
Gene_153,9.70693,9.00379,9.43173,9.30213,9.33787,8.0602,9.30329,8.89256
Gene_154,11.09227,10.00174,10.66424,10.58052,10.65135,9.87044,10.85977,10.82541
Gene_155,6.30963,5.7141,6.46201,6.58782,6.32688,5.72968,6.15887,6.584
Gene_156,6.90647,7.41956,6.80043,7.80433,6.798,7.00752,6.94112,7.72469
Gene_157,8.07428,8.90979,7.63133,8.6929,7.09315,9.10069,8.47924,8.72806
Gene_158,8.56243,9.03711,9.12553,8.75183,9.21068,9.0687,7.71056,9.11102
Gene_159,7.78978,7.56307,8.33436,9.30437,8.10757,8.92557,8.28905,8.40531
Gene_160,9.27313,9.32805,10.29383,9.26152,10.02208,9.37658,9.46262,9.73399
Gene_161,9.78312,8.90625,9.87751,9.76733,9.18941,9.48888,9.5362,9.34289
Gene_162,6.72728,6.35386,6.55661,5.54855,6.3252,6.68986,6.66603,6.60549
Gene_163,11.10484,11.01247,10.07898,10.54374,10.49869,11.09601,10.85911,10.89948
Gene_164,7.95481,7.94053,8.69638,9.05466,9.23525,8.49885,9.0366,8.90203
Gene_165,9.41027,10.12547,10.21754,10.04604,8.57312,9.42774,9.96434,10.4965
Gene_166,7.27383,7.23118,7.87731,6.62559,7.37684,6.87154,7.7095,7.60781
Gene_167,10.41107,10.97668,9.72323,9.91476,9.40988,10.84196,10.52342,10.29565
Gene_168,9.39097,9.68203,9.82857,9.65047,8.9461,9.45396,8.74964,9.81137
Gene_169,8.65648,8.65926,8.64899,8.79186,8.90115,8.06933,9.09119,8.36831
Gene_170,6.72216,6.93521,8.21471,7.442,6.61364,6.99893,7.10356,7.10846
Gene_171,10.95706,10.8974,10.63887,10.74452,11.13989,11.41753,10.88368,10.61916
Gene_172,9.61921,8.71898,9.89538,9.29274,9.03229,9.84864,9.49137,9.37184
Gene_173,6.09143,6.20783,5.49585,5.67112,6.7962,7.06115,6.51774,5.72634
Gene_174,10.88107,10.35712,11.12464,9.87177,10.59591,10.70657,10.45655,11.01883
Gene_175,8.04573,8.48611,7.95233,8.2496,8.1669,8.08372,8.64577,7.52514
Gene_176,7.80811,7.00084,7.12642,7.61546,6.27405,7.36475,7.4073,7.44567
Gene_177,7.58606,8.05995,8.52975,6.59614,8.00005,7.74178,8.24771,7.4839
Gene_178,9.83509,9.88898,10.25408,9.83829,9.8733,9.62408,8.8032,9.9342
Gene_179,8.91004,8.45696,8.74561,8.8477,9.14925,9.0284,8.07947,8.98579
Gene_180,6.65045,6.16286,6.56078,6.71617,6.98349,5.77795,6.57064,6.71212
Gene_181,7.65175,7.88718,7.11165,6.66866,6.84199,6.49,7.11294,6.78489
Gene_182,6.92909,7.56505,6.42169,7.13298,7.07879,7.67353,6.90924,6.68284
Gene_183,8.28617,7.95044,7.67014,8.16809,7.40776,7.95709,7.99132,7.68883
Gene_184,8.77788,9.86054,8.96966,9.67647,8.70159,8.97738,9.01544,9.51556
Gene_185,7.52586,7.63813,8.11337,7.33273,7.38786,7.37721,6.47384,6.73037
Gene_186,9.32841,8.4941,8.98211,9.07859,9.10305,10.47308,9.39265,9.87953
Gene_187,9.65852,10.34013,9.72368,10.07736,9.85692,9.67822,9.47102,9.72751
Gene_188,6.56628,6.20049,5.63708,6.80885,6.39688,6.01766,6.71346,5.88886
Gene_189,8.79812,8.0751,7.24396,7.28966,8.64429,7.64036,8.59042,8.0789
Gene_190,11.13759,10.74818,10.81253,11.67901,10.54881,11.46049,11.05137,11.17474
Gene_191,6.49991,6.58009,7.11803,6.98936,6.69624,7.30757,7.00038,6.36638
Gene_192,9.36258,8.64475,8.91369,8.51867,8.12344,7.97189,8.66659,9.06717
Gene_193,6.94523,8.21418,7.82379,7.82456,7.89333,8.58649,7.82952,7.0117
Gene_194,10.8874,10.86673,10.19881,10.39838,10.91812,9.97878,10.7675,9.93942
Gene_195,6.92848,7.20303,7.28168,6.39884,6.83687,6.42736,6.3915,6.48499
Gene_196,9.86894,9.71937,9.82945,9.59374,10.85091,9.84205,9.26065,10.23487
Gene_197,7.87325,6.84751,7.25668,7.13858,7.17567,6.97129,7.9347,7.80677
Gene_198,9.32802,9.4224,9.37162,9.48898,8.5909,9.732,8.84426,9.55808
Gene_199,7.87178,7.71512,7.36157,6.60775,7.30893,7.31057,6.31918,6.71144
Gene_200,7.39598,7.54282,7.21374,7.02596,6.76544,6.57488,7.26042,7.01801
Gene_201,7.63158,8.06254,7.88907,7.82786,7.69508,8.00415,7.16648,8.39107
Gene_202,8.39658,7.98921,7.53177,8.7901,8.46223,8.96459,7.64431,8.93181
Gene_203,5.98828,5.44677,6.99278,6.17492,5.94884,5.74198,5.94249,6.41036
Gene_204,6.58413,6.17533,7.39352,6.09732,6.12702,6.59147,6.16518,6.21461
Gene_205,6.73237,5.76614,7.29376,7.20529,6.71217,6.46859,7.33589,7.62581
Gene_206,6.08975,6.37052,6.4978,6.24384,7.00455,6.52906,6.12234,7.26033
Gene_207,9.38594,9.73034,9.38145,9.51394,9.29077,10.04931,9.77159,9.0116
Gene_208,8.04184,8.46718,9.0942,8.16156,8.76023,8.43448,8.41923,8.20772
Gene_209,11.18745,10.27634,10.08312,10.72184,10.28792,10.07491,10.23413,10.52691
Gene_210,7.07665,6.97794,5.99766,6.18048,6.41497,5.46344,6.54353,6.89991
Gene_211,7.82672,7.11042,7.75158,7.64587,8.41443,8.27251,7.14506,7.33383
Gene_212,9.71119,9.82636,8.84347,8.24639,9.51904,9.74329,8.7499,8.37309
Gene_213,8.81532,8.88372,8.60483,8.46441,8.92309,8.98721,8.89274,8.58161
Gene_214,9.60081,9.50761,9.11058,9.36996,9.38672,9.4447,10.46703,9.35534
Gene_215,8.79444,8.88043,8.46322,9.32406,8.04699,8.38045,8.40007,8.69503
Gene_216,6.92222,7.39516,7.71707,5.97648,7.23684,7.32439,6.50778,7.04635
Gene_217,8.70146,9.26201,8.70707,9.33524,9.01807,9.04422,9.09644,8.66755
Gene_218,7.2769,7.81095,6.92627,7.21147,7.52073,6.83639,7.74664,7.67273
Gene_219,6.60053,7.73341,6.71449,6.96946,7.18577,7.47211,6.00439,6.64505
Gene_220,8.54916,8.06233,9.02285,8.68433,8.15076,8.57325,8.77583,8.27343
Gene_221,6.28106,6.14536,6.78496,6.18644,6.17271,6.61525,6.66873,6.49631
Gene_222,8.45029,9.0942,8.495,8.20414,9.244,9.52472,9.01113,8.4799
Gene_223,10.37423,10.12795,10.27275,9.49918,9.83933,9.5062,9.33498,10.27467
Gene_224,11.03939,9.29495,9.89722,9.79471,9.73371,10.20921,9.91011,9.64234
Gene_225,7.48625,7.7724,7.50706,8.16257,8.83875,6.95391,7.91837,6.87965
Gene_226,10.51902,9.90696,9.99538,9.95281,8.92816,9.00294,9.71258,9.06564
Gene_227,7.86587,6.98372,7.40925,6.81752,7.38634,7.52967,6.91177,7.41596
Gene_228,8.03619,7.3089,8.04314,8.04572,7.52437,7.7528,8.07008,7.62421
Gene_229,9.28505,9.38264,8.80351,9.63848,8.97237,8.22307,9.22769,8.67135
Gene_230,8.43457,8.88177,8.62219,8.15644,7.69703,7.77075,8.3013,8.02849
Gene_231,9.3123,9.59438,9.43326,9.57921,10.65955,9.92064,9.59177,9.64481
Gene_232,6.74896,6.78667,6.73806,7.18243,6.20584,6.74628,7.2994,7.26779
Gene_233,9.14221,8.42981,9.17832,9.30231,8.88808,8.94613,8.882,8.84042
Gene_234,5.53204,6.31243,6.01103,6.08753,6.03547,5.58663,5.45095,6.17079
Gene_235,10.47206,10.04064,9.68358,10.11896,9.93899,9.92613,10.53853,9.81874
Gene_236,9.12986,9.84229,9.0633,9.27076,9.68753,9.41658,9.53747,9.35349
Gene_237,7.42303,6.35967,7.10299,7.13761,5.98469,7.44316,7.19694,7.66774
Gene_238,7.44571,7.79988,7.58141,7.38102,8.26272,8.00641,7.24775,7.68484
Gene_239,9.38338,8.54313,8.6912,8.4617,8.51955,8.81204,8.25517,8.43031
Gene_240,10.09121,10.42312,9.93554,10.88077,11.1063,10.13551,10.64098,10.01137
", row.names = 1, check.names = FALSE))
metadata <- read.csv(text = "sample,condition,batch,unit
C1,control,A,independent_C1
C2,control,B,independent_C2
C3,control,A,independent_C3
C4,control,B,independent_C4
T1,treatment,A,independent_T1
T2,treatment,B,independent_T2
T3,treatment,A,independent_T3
T4,treatment,B,independent_T4
", stringsAsFactors = FALSE)
metadata$condition <- factor(metadata$condition, levels = c("control", "treatment"))
stopifnot(identical(colnames(expr), metadata$sample), nrow(expr) == 240, ncol(expr) == 8, all(is.finite(expr)))
write.csv(expr, file.path(result_dir, "toy_log2_expression.csv"))
write.csv(metadata, file.path(result_dir, "toy_sample_metadata.csv"), row.names = FALSE)
print(dim(expr)); print(head(expr)); print(metadata)

### A. Data and design — 3 points
Use the metadata and the first rows of the matrix.

| Item | Your response |
|---|---|
| Give the number of genes, samples, and independent replicates per condition. | Replace with your answer. |
| State the input units and explain why applying another log transform is inappropriate. | Replace with your answer. |
| Are batches balanced across conditions? Why would complete confounding be a problem? | Replace with your answer. |

## B. Sample PCA — 5 points
PCA treats eight samples as observations and 240 genes as features. Center genes; do not standardize genes to unit variance for this baseline analysis.

### Your code edit: scaling
In the next cell find `scale_genes <- FALSE  # EDIT THIS LINE`. Run once with **FALSE**, predict how equalizing gene variances could affect the plot, change **FALSE to TRUE**, and rerun **that cell only**. Compare the sample relationships and PC1/PC2 percentages. Record both results in B's table, then restore **FALSE** and rerun. Keep the original `expr` matrix, sample labels, and centering fixed. This tests a preprocessing choice; it does not change the original expression values.

In [ ]:
scale_genes <- FALSE  # EDIT THIS LINE: FALSE -> TRUE -> restore FALSE
pca <- prcomp(t(expr), center = TRUE, scale. = scale_genes)
variance <- pca$sdev^2 / sum(pca$sdev^2)
cols <- ifelse(metadata$condition == "treatment", "#c64726", "#225ea8")
plot_pca <- function() {
  plot(pca$x[,1], pca$x[,2], col=cols, pch=19,
       xlab=sprintf("PC1 (%.1f%%)",100*variance[1]), ylab=sprintf("PC2 (%.1f%%)",100*variance[2]), main="Sample PCA: simulated log2 expression")
  text(pca$x[,1],pca$x[,2],labels=metadata$sample,pos=3,cex=.8)
  legend("topright",legend=c("Control","Treatment"),col=c("#225ea8","#c64726"),pch=19,bty="n")
}
plot_pca()
png(file.path(result_dir,"sample_PCA.png"),1000,700,res=120); plot_pca(); dev.off()
print(round(100*variance,2))
write.csv(pca$x, file.path(result_dir,"PCA_scores.csv"))
write.csv(data.frame(PC=seq_along(variance),fraction=variance),file.path(result_dir,"PCA_variance.csv"),row.names=FALSE)
B_trial <- data.frame(scale_genes=scale_genes,PC1=variance[1],PC2=variance[2])
B_file <- file.path(result_dir,"B_PCA_edit_history.csv")
write.table(B_trial,B_file,sep=",",row.names=FALSE,col.names=!file.exists(B_file),append=file.exists(B_file))

### B. Response
PCA axes may reverse sign without changing the relationships.

| Item | Your response |
|---|---|
| Report PC1 and PC2 variance percentages; describe treatment/control relationships and one sample relationship. | Replace with your answer. |
| Does separation prove a treatment effect or causation? Explain what independent replication and design add. | Replace with your answer. |
| What change in weighting would scale.=TRUE make? | Replace with your answer. |

| Manual trial | Prediction before running | PC1 and PC2 percentages | Observed sample relationship |
|---|---|---|---|
| scale_genes <- FALSE | Replace with prediction. | Replace with output. | Replace with observation. |
| scale_genes <- TRUE | Replace with prediction. | Replace with output. | Replace with observation. |

Restore FALSE before your final saved run. Explain one difference or similarity using the plots, not a required PCA-axis sign.

## C. Effect size and multiple testing — 6 points
The effect is mean(log2 treatment) − mean(log2 control), i.e., the log2 ratio of geometric means on the original expression scale. It is not the log2 ratio of arithmetic means. Use two-sided unpaired Welch tests; the toy experiment has independent samples, not matched pairs.

In [ ]:
control <- metadata$condition == "control"
treated <- metadata$condition == "treatment"
log2FC <- rowMeans(expr[,treated,drop=FALSE]) - rowMeans(expr[,control,drop=FALSE])
pvalue <- apply(expr,1,function(x) t.test(x[treated],x[control],paired=FALSE,var.equal=FALSE)$p.value)
padj <- p.adjust(pvalue,method="BH")
results <- data.frame(gene=rownames(expr),mean_log2=rowMeans(expr),log2FC=log2FC,pvalue=pvalue,padj=padj,row.names=NULL)
stopifnot(all(results$padj >= results$pvalue - 1e-12),all(results$padj >= 0 & results$padj <= 1))
print(head(results[order(results$padj),],10))
cat("Raw p <= .05:",sum(results$pvalue<=.05),"; BH adjusted p <= .05:",sum(results$padj<=.05),"
")
write.csv(results,file.path(result_dir,"differential_expression_results.csv"),row.names=FALSE)

### C. Response
Use the calculated results; gene names have no known biological function.

| Item | Your response |
|---|---|
| Interpret log2FC +2 and −2 on the original scale and explain the sign convention. | Replace with your answer. |
| Report raw-p and BH-adjusted-p hit counts and explain their difference. | Replace with your answer. |
| Explain FDR in your own words. Is adjusted p the probability an individual gene is false? | Replace with your answer. |
| Why would a large effect with highly variable replicates have weak statistical evidence? | Replace with your answer. |

## C continued — Generate toy data and change the scatter yourself
This **separate controlled simulation** lets you change the data generator. It does not overwrite the fixed CSV dataset or earlier `results`. Each run recreates the same random deviations from the same seed. The left volcano panel is always the fixed reference; the right uses your edited values. The printed summary also shows BH-only hit counts and a simulated gene's replicate spread.

Edit only the three assignments marked `# EDIT THIS LINE` in the next cell. In R, change the number after `<-`, leaving the name and comment intact. Use its Run button or Shift+Enter to run **that cell only** during each comparison.

1. **Reference:** run `noise_sd <- 0.35`, `effect_size <- 2`, `n_per_condition <- 4`. The two panels should match.
2. **More scatter:** predict first. Change only **noise_sd from 0.35 to 1.2**, rerun, and record what happens to replicate spread, effect estimates, adjusted p-values, and hits. This multiplies the standard deviation of the random expression deviations; it does not add more samples.
3. **Smaller true effect:** restore **noise_sd to 0.35**. Change only **effect_size from 2 to 0.5**, predict, rerun, and record the result. Compare BH-only hits with those that also pass the |effect|≥1 rule.
4. **More replicates:** restore **effect_size to 2**. Change only **n_per_condition from 4 to 8**, predict, rerun, and compare. The first four samples of each condition are retained, and four additional independent simulated samples are added to each; this is not copying rows to pretend there are more biological replicates.
5. **Restore:** set all three values to **0.35, 2, 4**, rerun, and verify the reference returns. Keep the seed, number of genes, group-effect pattern, alpha=0.05, and |effect| cutoff=1 fixed.

The simulator uses 240 artificial genes: 30 with positive, 30 with negative, and 180 with zero true treatment effects. Values are continuous log2 expression, not RNA-seq counts. Each realized run can depart from the typical expectation; report what you actually observe. Gene-wise tests do not borrow information across genes as specialist omics methods can.

In [ ]:
# CHANGE THESE THREE MARKED LINES ONE AT A TIME.
noise_sd <- 0.35       # EDIT THIS LINE: 0.35 -> 1.2 -> restore 0.35
effect_size <- 2       # EDIT THIS LINE: 2 -> 0.5 -> restore 2
n_per_condition <- 4   # EDIT THIS LINE: 4 -> 8 -> restore 4

# HOLD THE REST OF THIS CELL FIXED during the comparisons.
stopifnot(noise_sd > 0, effect_size >= 0, n_per_condition %in% 2:8)
set.seed(1115)  # HOLD FIXED: identical underlying random draws on every run.
sim_errors <- matrix(rnorm(240*16),nrow=240,ncol=16)
sim_means <- seq(6,11,length.out=240)
simulate_expression <- function(sd_value,effect_value,n_value) {
  effects <- c(rep(effect_value,30),rep(-effect_value,30),rep(0,180))
  # Controls occupy the first 8 random columns; treatment the next 8.
  used <- c(seq_len(n_value),8+seq_len(n_value))
  A <- matrix(sim_means,nrow=240,ncol=2*n_value) + sd_value*sim_errors[,used,drop=FALSE]
  A[,(n_value+1):(2*n_value)] <- A[,(n_value+1):(2*n_value),drop=FALSE] + effects
  rownames(A) <- sprintf("Sim_%03d",seq_len(240))
  colnames(A) <- c(paste0("C",seq_len(n_value)),paste0("T",seq_len(n_value)))
  A
}
analyze_simulation <- function(A,n_value) {
  C <- seq_len(n_value); T <- n_value+seq_len(n_value)
  p <- apply(A,1,function(x) t.test(x[T],x[C],var.equal=FALSE)$p.value)
  data.frame(gene=rownames(A),log2FC=rowMeans(A[,T,drop=FALSE])-rowMeans(A[,C,drop=FALSE]),padj=p.adjust(p,"BH"))
}
sim_baseline <- simulate_expression(.35,2,4)
sim_edited <- simulate_expression(noise_sd,effect_size,n_per_condition)
sim_baseline_results <- analyze_simulation(sim_baseline,4)
sim_edited_results <- analyze_simulation(sim_edited,n_per_condition)
sim_summary <- function(label,A,R,n_value) {
  data.frame(setting=label,replicates_per_condition=n_value,
    gene1_control_SD=sd(A[1,seq_len(n_value)]),gene1_treatment_SD=sd(A[1,n_value+seq_len(n_value)]),
    gene1_estimated_effect=R$log2FC[1],gene1_adjusted_p=R$padj[1],
    BH_only_hits=sum(R$padj<=.05),BH_and_effect_hits=sum(R$padj<=.05 & abs(R$log2FC)>=1))
}
sim_comparison <- rbind(sim_summary("Fixed reference",sim_baseline,sim_baseline_results,4),sim_summary("Your edit",sim_edited,sim_edited_results,n_per_condition))
print(sim_comparison)
sim_ymax <- max(-log10(pmax(c(sim_baseline_results$padj,sim_edited_results$padj),.Machine$double.xmin)))*1.15
sim_xmax <- max(abs(c(sim_baseline_results$log2FC,sim_edited_results$log2FC)))*1.15
plot_sim_comparison <- function() {
  par(mfrow=c(1,2))
  for (j in 1:2) {
    R <- if(j==1) sim_baseline_results else sim_edited_results
    category <- ifelse(R$padj<=.05 & R$log2FC>=1,"Up",ifelse(R$padj<=.05 & R$log2FC<= -1,"Down","Other"))
    palette <- c(Up="#c64726",Down="#225ea8",Other="#999999")
    plot(R$log2FC,-log10(pmax(R$padj,.Machine$double.xmin)),pch=19,col=palette[category],
      xlim=c(-sim_xmax,sim_xmax),ylim=c(0,sim_ymax),xlab="Treatment minus control log2 effect",ylab="-log10(BH-adjusted p)",
      main=if(j==1) "Fixed reference" else sprintf("Edited: SD=%g, effect=%g, n=%d",noise_sd,effect_size,n_per_condition))
    abline(h=-log10(.05),v=c(-1,1),lty=2)
  }
  par(mfrow=c(1,1))
}
plot_sim_comparison()
sim_tag <- sprintf("sd_%g_effect_%g_n_%d",noise_sd,effect_size,n_per_condition)
png(file.path(result_dir,paste0("C_guided_",sim_tag,".png")),1400,700,res=120); plot_sim_comparison(); dev.off()
write.csv(sim_edited,file.path(result_dir,paste0("C_simulated_expression_",sim_tag,".csv")))
write.csv(sim_edited_results,file.path(result_dir,paste0("C_simulated_results_",sim_tag,".csv")),row.names=FALSE)
sim_trial <- cbind(data.frame(noise_sd=noise_sd,effect_size=effect_size,n_per_condition=n_per_condition),sim_comparison[2,,drop=FALSE])
sim_file <- file.path(result_dir,"C_simulation_edit_history.csv")
write.table(sim_trial,sim_file,sep=",",row.names=FALSE,col.names=!file.exists(sim_file),append=file.exists(sim_file))
cat("Record your prediction and observation below before your next edit.\n")

### C continued — Your generator-edit record (within C's 6 points)
| Manual setting: SD, effect, n per condition | Prediction BEFORE running | BH-only / BH-and-effect hit counts | Gene 1 spread and adjusted p; what changed and why? |
|---|---|---|---|
| Reference: 0.35, 2, 4 | Replace with prediction. | Replace with output. | Replace with observation. |
| More scatter: 1.2, 2, 4 | Replace with prediction. | Replace with output. | Replace with observation. |
| Smaller effect: 0.35, 0.5, 4 | Replace with prediction. | Replace with output. | Replace with observation. |
| More replicates: 0.35, 2, 8 | Replace with prediction. | Replace with output. | Replace with observation. |
| Restored: 0.35, 2, 4 | Predict what should return. | Replace with output. | Explain why holding the seed fixed made comparison easier. |

Use the two volcano panels and the printed summary together. Explain why stronger statistical evidence and passing a chosen magnitude cutoff are different questions. If a change did not affect hit counts in this run, describe what else changed; do not invent the expected pattern.

## D. Volcano plots and thresholds — 8 points
The vertical axis is **−log10(BH-adjusted p)** in this notebook; many plots instead use raw p, so always read the label. We floor only the plotted values at machine minimum to avoid log10(0), leaving stored p-values unchanged. Color uses both adjusted-p and effect-size cutoffs. This is a classroom prioritization rule, not a test that the true effect exceeds the cutoff.

### Your code edit: the effect cutoff
Find `lfc_cutoff <- 1  # EDIT THIS LINE` in the next cell. Run at **1**, predict what will happen, change to **0.5**, and rerun **that cell only**. Record the Up/Down/Other counts. Change to **1.5**, predict and rerun, then restore **1** and rerun. Keep `alpha <- 0.05`, the input data, and the already computed p-values fixed. You must edit and rerun this line yourself; the supplied automatic comparison panels do not replace this step. A cutoff changes the decision colors, not the underlying p-values.

In [ ]:
alpha <- 0.05  # HOLD FIXED for this comparison
lfc_cutoff <- 1  # EDIT THIS LINE: 1 -> 0.5 -> 1.5 -> restore 1
classify <- function(cutoff) {
  ifelse(results$padj <= alpha & results$log2FC >= cutoff,"Up",
         ifelse(results$padj <= alpha & results$log2FC <= -cutoff,"Down","Other"))
}
plot_volcano <- function(cutoff) {
  category <- classify(cutoff)
  palette <- c(Up="#c64726",Down="#225ea8",Other="#999999")
  y <- -log10(pmax(results$padj,.Machine$double.xmin))
  plot(results$log2FC,y,pch=19,col=palette[category],xlab="Treatment − control log2 effect",ylab="−log10(BH-adjusted p)",ylim=c(0,max(y)*1.2),main=paste("Volcano: |log2 effect| cutoff",cutoff))
  abline(h=-log10(alpha),v=c(-cutoff,cutoff),lty=2)
  top <- match(c("Gene_001","Gene_041","Gene_069"),results$gene)
  text(results$log2FC[top],y[top],results$gene[top],pos=3,cex=.7)
  legend("topright",legend=names(palette),col=palette,pch=19,bty="n")
}
plot_volcano(lfc_cutoff)
png(file.path(result_dir,"volcano_default.png"),1000,750,res=120); plot_volcano(lfc_cutoff); dev.off()
png(file.path(result_dir,"volcano_cutoff_comparison.png"),1400,650,res=120); par(mfrow=c(1,2)); plot_volcano(.5); plot_volcano(1.5); dev.off()
par(mfrow=c(1,2)); plot_volcano(.5); plot_volcano(1.5); par(mfrow=c(1,1))
threshold_counts <- do.call(rbind,lapply(c(.5,1,1.5),function(v) data.frame(cutoff=v,Up=sum(classify(v)=="Up"),Down=sum(classify(v)=="Down"),Other=sum(classify(v)=="Other"))))
print(threshold_counts)
results$category <- classify(lfc_cutoff)
write.csv(threshold_counts,file.path(result_dir,"threshold_counts.csv"),row.names=FALSE)
write.csv(results,file.path(result_dir,"classified_results.csv"),row.names=FALSE)
D_trial <- data.frame(lfc_cutoff=lfc_cutoff,alpha=alpha,Up=sum(classify(lfc_cutoff)=="Up"),Down=sum(classify(lfc_cutoff)=="Down"),Other=sum(classify(lfc_cutoff)=="Other"))
print(D_trial)
D_file <- file.path(result_dir,"D_threshold_edit_history.csv")
write.table(D_trial,D_file,sep=",",row.names=FALSE,col.names=!file.exists(D_file),append=file.exists(D_file))

### D. Response
Read points using both axes; cite actual gene IDs and values.

| Item | Your response |
|---|---|
| Explain the upper-right, upper-left, and lower portions of the volcano plot. | Replace with your answer. |
| Choose one Up, one Down, and one Other gene; report their log2FC and adjusted p and explain each classification. | Replace with your answer. |
| Report counts at effect cutoffs .5, 1, 1.5. Which evidence stays unchanged when the visual threshold changes? | Replace with your answer. |
| Explain why a small-effect gene can have strong statistical evidence, and why changing thresholds after seeing a plot needs care. | Replace with your answer. |

| Manually entered cutoff | Prediction before running | Up / Down / Other counts from your run | What stayed fixed? |
|---|---|---|---|
| 1 | Replace with prediction. | Replace with counts. | Replace with answer. |
| 0.5 | Replace with prediction. | Replace with counts. | Replace with answer. |
| 1.5 | Replace with prediction. | Replace with counts. | Replace with answer. |

Restore 1 and rerun before continuing to E.

## E. MA plot and replicate check — 5 points
The MA-style plot uses average log2 expression on x and treatment-minus-control log2 effect on y. Contrast its question with the volcano plot. Then inspect individual replicate values rather than only a group average.

In [ ]:
plot_ma <- function() {
  palette <- c(Up="#c64726",Down="#225ea8",Other="#999999")
  plot(results$mean_log2,results$log2FC,pch=19,col=palette[results$category],xlab="Mean log2 expression",ylab="Treatment − control log2 effect",main="MA-style view")
  abline(h=0,lty=2)
}
plot_ma(); png(file.path(result_dir,"MA_plot.png"),1000,700,res=120); plot_ma(); dev.off()
# A stable positive-effect example and a deliberately variable example.
selected <- c("Gene_001","Gene_081")
plot_replicates <- function() {
  par(mfrow=c(1,2))
  for(g in selected) {
    stripchart(as.numeric(expr[g,]) ~ metadata$condition,vertical=TRUE,method="stack",pch=19,
               col=c("#225ea8","#c64726"),ylab="Simulated log2 expression",main=g)
  }
  par(mfrow=c(1,1))
}
plot_replicates(); png(file.path(result_dir,"replicate_examples.png"),1100,650,res=120); plot_replicates(); dev.off()
print(results[results$gene %in% selected,])

### E. Response
Base your explanation on the MA and individual-replicate plots.

| Item | Your response |
|---|---|
| What relationship is visible in an MA plot that the volcano x-axis does not show? | Replace with your answer. |
| Compare Gene_001 and Gene_081: effect estimate, replicate spread, and adjusted p. Does a large effect guarantee a hit? | Replace with your answer. |
| Give one next step needed before claiming an actual gene has a biological role; explain why a volcano plot alone cannot supply that role. | Replace with your answer. |

### F. Reproducibility and limits — 3 points
Finish and save the notebook.

| Item | Your response |
|---|---|
| Record input units, contrast, test, adjustment method, and final thresholds. | Replace with your answer. |
| Explain why this toy workflow must be changed for raw RNA-seq counts, and identify one limitation of the simulation. | Replace with your answer. |
| Disclose any AI help (or state none), code changes, and a check you made against the computed results. | Replace with your answer. |

Briefly name one R ecosystem strength that could help a real gene-expression project and one task for which you would still use Python. Confirm your B, C-continued, and D manual parameter edits and any further changes.

In [ ]:
record <- c("Assignment 11: R gene-expression toy exercise", "Data: 240 genes x8 independent simulated samples; four per condition", "Units: simulated normalized log2 expression; NOT raw counts", "Data SHA256: ada25bd3acd52adda2856490fbe6a98f8c070ce2157ed7b5c5314f8e8baad38d", "Generator: NumPy default_rng(1115); fixed data embedded in notebook", "Contrast: mean log2 treatment minus mean log2 control", "Test: unpaired two-sided Welch t-test per gene", "Adjustment: BH across all 240 genes", paste("alpha:",alpha,"; effect cutoff:",lfc_cutoff), paste("PCA: samples as observations; genes centered; scale_genes:",scale_genes), paste("Guided simulation noise_sd:",noise_sd,"effect_size:",effect_size,"n_per_condition:",n_per_condition))
writeLines(record,file.path(result_dir,"analysis_record.txt"))
capture.output(sessionInfo(),file=file.path(result_dir,"R_session_info.txt"))
archive <- "assignment_11_results.zip"
if (file.exists(archive)) file.remove(archive)
zip(archive,files=list.files(result_dir,full.names=TRUE),flags="-j")
stopifnot(file.exists(archive),file.info(archive)$size > 0)
cat("Saved",archive,". In Colab, open the Files panel, refresh, then download this ZIP.\n")

## Submit and check
1. Save the notebook with all plots visible and A–F response tables completed. Download the executed `.ipynb` from Colab’s File menu.
2. Download **assignment_11_results.zip** from the Colab Files panel. It contains input CSVs, DE results, threshold counts, PCA tables, plots, analysis record, and R session information.
3. Upload the notebook and ZIP together to Canvas. No separate Word report or screenshot-only submission.

**Rubric: A 3 + B 5 + C 6 + D 8 + E 5 + F 3 = 30 points.** Full credit requires correct calculations plus explanations grounded in your outputs; incomplete explanations receive partial credit.

References: [R t.test](https://stat.ethz.ch/R-manual/R-devel/library/stats/html/t.test.html), [R p.adjust](https://stat.ethz.ch/R-manual/R-devel/library/stats/html/p.adjust.html), [DESeq2 RNA-seq count workflow for comparison](https://www.bioconductor.org/packages/release/bioc/vignettes/DESeq2/inst/doc/DESeq2.html).

Guided edits are part of existing B (5), C (6), D (8), and F (3) points; **total remains 30**. Include completed prediction/observation tables and code with its marked lines. After the required trials, restore `scale_genes <- FALSE`, `noise_sd <- 0.35`, `effect_size <- 2`, `n_per_condition <- 4`, and `lfc_cutoff <- 1`. Run all once to refresh the final ZIP, then save/download your executed notebook. Trial CSVs append during the same runtime; duplicate rows from reruns are acceptable. If the runtime restarts, redo the trials before exporting because its working files are lost.